# Notebook 01: Bulk Data Acquisition

This notebook downloads and caches multi-season Formula 1 data using FastF1.

We are building a robust raw dataset for machine learning with these outputs:

1. Event calendar and session queue
2. Session-level results table
3. Lap-level timing table
4. Per-lap telemetry feature table
5. Resumable extraction log
6. Combined master tables

Important guardrails for this notebook:

- We collect data only (no train/test split here).
- We keep season and round metadata for future walk-forward validation.
- FastF1 still doesnt have MOM data so i just used DRS for training
- We run a DRY_RUN first, then full extraction.


In [11]:
# import dependencies
from pathlib import Path
from datetime import datetime, timezone
import re
import time
import warnings

import numpy as np
import pandas as pd
import fastf1
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print("Imports completed.")
print("FastF1 version:", fastf1.__version__)

Imports completed.
FastF1 version: 3.8.2


## Step 0: Configuration and Paths

In this step we define:

- Seasons and session types to download
- DRY_RUN controls
- Telemetry extraction behavior
- Output formats (Parquet + CSV)
- Resume and retry behavior
- Folder paths and FastF1 cache


In [12]:
# Core extraction scope
SEASONS = [2023, 2024, 2025]
SESSION_CODES = ["Q", "R"]

# Safe testing mode
DRY_RUN = True
MAX_EVENTS_PER_SEASON = 2  # used only when DRY_RUN=True

# Output controls
EXTRACT_TELEMETRY = True
SAVE_PARQUET = True
SAVE_CSV = True

# Reliability controls
RESUME_MODE = True
RETRY_ERRORS = True
MAX_RETRIES = 3
SLEEP_BETWEEN_RETRIES_SECONDS = 2

# Resolve project root even when launching from notebooks folder
project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
cache_dir = raw_dir / "fastf1_cache"

results_dir = raw_dir / "session_results"
laps_dir = raw_dir / "session_laps"
telemetry_dir = raw_dir / "session_telemetry_features"
logs_dir = raw_dir / "logs"

for folder in [raw_dir, processed_dir, cache_dir, results_dir, laps_dir, telemetry_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# FastF1 cache
fastf1.Cache.enable_cache(str(cache_dir))

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
SESSION_QUEUE_PATH = logs_dir / "session_queue.csv"
EXTRACTION_LOG_PATH = logs_dir / "extraction_log.csv"

print("Project root:", project_root)
print("Cache dir:", cache_dir)
print("Session scope:", SESSION_CODES)
print("DRY_RUN:", DRY_RUN)
print("Run timestamp:", RUN_TS)

Project root: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction
Cache dir: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\fastf1_cache
Session scope: ['Q', 'R']
DRY_RUN: True
Run timestamp: 2026-04-12 19:21:40 UTC


## Step 1: Download Event Calendar

We fetch schedules for each target season and keep championship rounds only.

If DRY_RUN is enabled, we keep only the first few events per season so we can validate the pipeline quickly.


In [13]:
all_event_frames = []
for season in SEASONS:
    print(f"\nLoading calendar for {season}...")
    try: 
        season_events = fastf1.get_event_schedule(season, include_testing=False)
    except TypeError:
        # fallback for fastf1 versions without include testing args
        season_events = fastf1.get_event_schedule(season)

    season_events = season_events.copy()
    season_events["Season"] = season

    # keep championship rounds only
    if "RoundNumber" in season_events.columns:
        season_events = season_events[season_events["RoundNumber"].notna()]
        season_events = season_events[season_events["RoundNumber"] > 0]
        season_events["RoundNumber"] = season_events["RoundNumber"].astype(int)
        season_events = season_events.sort_values("RoundNumber")

    # for dry run
    if DRY_RUN:
        season_events = season_events.head(MAX_EVENTS_PER_SEASON)
    
    all_event_frames.append(season_events)
    print(f"Events kept for {season}: {len(season_events)}")

    events_df = pd.concat(all_event_frames, ignore_index=True)


    calendar_cols = ["Season", "RoundNumber", "EventName", "Country", "Location", "EventDate", "EventFormat"]
    calendar_cols = [c for c in calendar_cols if c in events_df.columns]

    print("\nTotal events in this run:", len(events_df))
    display(events_df[calendar_cols].head(25))


Loading calendar for 2023...
Events kept for 2023: 2

Total events in this run: 2


,Season,RoundNumber,EventName,Country,Location,EventDate,EventFormat
0,2023,1,Bahrain Grand Prix,Bahrain,Sakhir,2023-03-05,conventional
1,2023,2,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2023-03-19,conventional



Loading calendar for 2024...
Events kept for 2024: 2

Total events in this run: 4


,Season,RoundNumber,EventName,Country,Location,EventDate,EventFormat
0,2023,1,Bahrain Grand Prix,Bahrain,Sakhir,2023-03-05,conventional
1,2023,2,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2023-03-19,conventional
2,2024,1,Bahrain Grand Prix,Bahrain,Sakhir,2024-03-02,conventional
3,2024,2,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2024-03-09,conventional



Loading calendar for 2025...
Events kept for 2025: 2

Total events in this run: 6


,Season,RoundNumber,EventName,Country,Location,EventDate,EventFormat
0,2023,1,Bahrain Grand Prix,Bahrain,Sakhir,2023-03-05,conventional
1,2023,2,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2023-03-19,conventional
2,2024,1,Bahrain Grand Prix,Bahrain,Sakhir,2024-03-02,conventional
3,2024,2,Saudi Arabian Grand Prix,Saudi Arabia,Jeddah,2024-03-09,conventional
4,2025,1,Australian Grand Prix,Australia,Melbourne,2025-03-16,conventional
5,2025,2,Chinese Grand Prix,China,Shanghai,2025-03-23,sprint_qualifying


## Step 2: Build Session Queue and Resume State

We convert each event into multiple jobs:

Event x SessionCode = one extraction job

Then we merge with extraction_log.csv (if it exists) so reruns can skip completed sessions.


In [14]:
def safe_name(text):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_")

# Build session queue
queue_rows = []
for _, row in events_df.iterrows():
    season = int(row["Season"])
    round_number = int(row["RoundNumber"]) if pd.notna(row["RoundNumber"]) else -1
    event_name = str(row["EventName"])
    event_name_safe = safe_name(event_name)
    event_date = pd.to_datetime(row["EventDate"], errors="coerce")

    for session_code in SESSION_CODES:
        session_key = f"{season}_R{round_number:02d}_{event_name_safe}_{session_code}"
        queue_rows.append(
            {
                "SessionKey": session_key,
                "Season": season,
                "RoundNumber": round_number,
                "EventName": event_name,
                "EventNameSafe": event_name_safe,
                "EventDate": event_date,
                "SessionCode": session_code,
            }
        )

queue_df = pd.DataFrame(queue_rows).sort_values(
    ["Season", "RoundNumber", "SessionCode"]
).reset_index(drop=True)

queue_df.to_csv(SESSION_QUEUE_PATH, index=False)
print("Session queue saved to:", SESSION_QUEUE_PATH)
print("Total jobs:", len(queue_df))

# Load or create extraction log
log_columns = [
    "SessionKey", "Season", "RoundNumber", "EventName", "SessionCode",
    "Status", "AttemptCount", "ResultsRows", "LapsRows", "TelemetryRows",
    "ElapsedSeconds", "UpdatedAtUTC", "ErrorMessage"
]

if RESUME_MODE and EXTRACTION_LOG_PATH.exists():
    extraction_log_df = pd.read_csv(EXTRACTION_LOG_PATH)
    extraction_log_df = extraction_log_df.drop_duplicates(subset=["SessionKey"], keep="last")
    print("Loaded existing extraction log:", EXTRACTION_LOG_PATH)
else:
    extraction_log_df = pd.DataFrame(columns=log_columns)
    print("Starting a new extraction log.")

status_lookup = (
    extraction_log_df[["SessionKey", "Status"]]
    if not extraction_log_df.empty
    else pd.DataFrame(columns=["SessionKey", "Status"])
)

queue_status_df = queue_df.merge(status_lookup, on="SessionKey", how="left")
queue_status_df["Status"] = queue_status_df["Status"].fillna("TODO")

if RETRY_ERRORS:
    todo_df = queue_status_df[queue_status_df["Status"].isin(["TODO", "ERROR"])].copy()
else:
    todo_df = queue_status_df[queue_status_df["Status"] == "TODO"].copy()

print("Already DONE:", int((queue_status_df["Status"] == "DONE").sum()))
print("Pending now:", len(todo_df))
display(queue_status_df["Status"].value_counts(dropna=False))
display(queue_df.groupby(["Season", "SessionCode"]).size().rename("Jobs").reset_index())

Session queue saved to: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\logs\session_queue.csv
Total jobs: 12
Starting a new extraction log.
Already DONE: 0
Pending now: 12


Status
TODO    12
Name: count, dtype: int64

,Season,SessionCode,Jobs
0,2023,Q,2
1,2023,R,2
2,2024,Q,2
3,2024,R,2
4,2025,Q,2
5,2025,R,2


## Step 3: Helper Functions

These functions handle:

- Timedelta conversion to seconds
- Metadata injection
- Results extraction
- Laps extraction with relative pace columns
- Telemetry aggregation per lap
- Dual-format saving
- Log row upsert


In [ ]:
def td_series_to_seconds(series):
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()

def save_table_dual(df, out_base_path):
    """
    Saves a DataFrame as parquet/csv depending on config flags.
    Returns list of saved paths.
    """
    saved_paths = []
    if df is None or len(df) == 0:
        return saved_paths

    if SAVE_PARQUET:
        parquet_path = Path(str(out_base_path) + ".parquet")
        df.to_parquet(parquet_path, index=False)
        saved_paths.append(parquet_path)

    if SAVE_CSV:
        csv_path = Path(str(out_base_path) + ".csv")
        df.to_csv(csv_path, index=False)
        saved_paths.append(csv_path)

    return saved_paths


def add_metadata(df, meta):
    for k, v in meta.items():
        df[k] = v
    return df


def build_results_table(session, meta):
    results_df = session.results.copy()
    if results_df is None or len(results_df) == 0:
        return pd.DataFrame()

    results_df = results_df.reset_index(drop=True)

    # convert key timing columns to numeric seconds for ML friendliness
    for col in ["Q1", "Q2", "Q3", "Time"]:
        if col in results_df.columns:
            results_df[f"{col}Seconds"] = td_series_to_seconds(results_df[col])

    results_df = add_metadata(results_df, meta)
    return results_df


def build_laps_table(session, meta):
    laps_df = session.laps.copy()
    if laps_df is None or len(laps_df) == 0:
        return pd.DataFrame()

    laps_df = laps_df.reset_index(drop=True)

    # convert timedeltas to seconds for easier downstream modeling
    td_cols = ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]
    for col in td_cols:
        if col in laps_df.columns:
            laps_df[f"{col}Seconds"] = td_series_to_seconds(laps_df[col])

    # relative pace features are safer across regulation changes than raw times alone
    if "LapTimeSeconds" in laps_df.columns:
        session_best = laps_df["LapTimeSeconds"].min(skipna=True)
        laps_df["DeltaToSessionBestLapSeconds"] = laps_df["LapTimeSeconds"] - session_best

        if "Driver" in laps_df.columns:
            driver_best = laps_df.groupby("Driver")["LapTimeSeconds"].transform("min")
            laps_df["DeltaToDriverBestLapSeconds"] = laps_df["LapTimeSeconds"] - driver_best

    laps_df = add_metadata(laps_df, meta)
    return laps_df


def aggregate_one_lap_telemetry(lap, meta):
    """
    Build one feature row from one lap's car telemetry.
    Dynamically sniffs for 2026 Manual Override Mode (MOM) or legacy DRS.
    """
    try:
        car = lap.get_car_data().copy()
    except Exception:
        return None

    if car is None or len(car) == 0:
        return None

    row = {
        "Driver": lap.get("Driver", np.nan),
        "DriverNumber": lap.get("DriverNumber", np.nan),
        "LapNumber": lap.get("LapNumber", np.nan),
        "Stint": lap.get("Stint", np.nan),
        "Compound": lap.get("Compound", np.nan),
        "TyreLife": lap.get("TyreLife", np.nan),
        "Samples": len(car),
    }

    numeric_cols = ["Speed", "Throttle", "Brake", "RPM", "nGear"]
    for col in numeric_cols:
        if col in car.columns:
            values = pd.to_numeric(car[col], errors="coerce")
            row[f"{col}Mean"] = float(values.mean())
            row[f"{col}Std"] = float(values.std())
            row[f"{col}Min"] = float(values.min())
            row[f"{col}Max"] = float(values.max())

    # ==========================================
    # 2026 OVERRIDE / DRS SNIFFER
    # ==========================================
    # TODO: Revisit in mid-2026. Check if future versions of fast f1 has implemented MOM / Active Aero.

    # we check multiple potential column names FOM might be using for the override boost
    boost_columns = ["DRS", "Override", "MOM", "ERS_Deploy"]
    
    for boost_col in boost_columns:
        if boost_col in car.columns:
            boost_values = pd.to_numeric(car[boost_col], errors="coerce")
            
            # Historically, values >= 8 meant DRS was active. 
            # We assume FOM maintains a similar logic for the Override signal.
            # We calculate the *percentage* of the lap the boost was active.
            active_samples = (boost_values >= 8).sum()
            total_samples = len(boost_values)
            
            # Create a generic "BoostActiveRatio" feature so XGBoost doesn't care if it's DRS or MOM
            row["BoostActiveRatio"] = float(active_samples / total_samples) if total_samples > 0 else 0.0
            
            # Once we find the valid boost column, we stop looking
            break 
    
    # If the API provided absolutely no boost data, default to 0
    if "BoostActiveRatio" not in row:
        row["BoostActiveRatio"] = 0.0

    # optional distance-derived feature
    try:
        car_dist = car.add_distance()
        if "Distance" in car_dist.columns:
            row["DistanceMax"] = float(pd.to_numeric(car_dist["Distance"], errors="coerce").max())
    except Exception:
        row["DistanceMax"] = np.nan

    lap_time_td = pd.to_timedelta(lap.get("LapTime", pd.NaT), errors="coerce")
    row["LapTimeSeconds"] = lap_time_td.total_seconds() if pd.notna(lap_time_td) else np.nan

    row.update(meta)
    return row


def build_telemetry_features_table(session, meta):
    if not EXTRACT_TELEMETRY:
        return pd.DataFrame()

    rows = []
    laps_obj = session.laps.copy()
    if laps_obj is None or len(laps_obj) == 0:
        return pd.DataFrame()

    for _, lap in laps_obj.iterlaps():
        feat_row = aggregate_one_lap_telemetry(lap, meta)
        if feat_row is not None:
            rows.append(feat_row)

    if len(rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(rows)

# delete old record and insert new one if a race is already in log
def upsert_log_row(log_df, row_dict):
    if log_df is None or len(log_df) == 0:
        return pd.DataFrame([row_dict])

    remaining = log_df[log_df["SessionKey"] != row_dict["SessionKey"]]
    return pd.concat([remaining, pd.DataFrame([row_dict])], ignore_index=True)

## Step 4: Main Extraction Loop

For each pending job:

1. Load FastF1 session data
2. Extract results, laps, telemetry features
3. Save session files
4. Update extraction_log.csv immediately

If one session fails, we log ERROR and continue.


In [16]:
overall_start = time.time()

# Make sure extraction_log_df exists even if earlier cell was skipped
if "extraction_log_df" not in globals():
    extraction_log_df = pd.DataFrame(columns=log_columns)

for i, job in enumerate(todo_df.itertuples(index=False), start=1):
    session_start = time.time()

    session_key = job.SessionKey
    season = int(job.Season)
    round_number = int(job.RoundNumber)
    event_name = str(job.EventName)
    session_code = str(job.SessionCode)

    print(f"\n[{i}/{len(todo_df)}] {session_key}")

    status = "ERROR"
    error_message = ""
    results_rows = 0
    laps_rows = 0
    telemetry_rows = 0
    attempt_count = 0

    for attempt in range(1, MAX_RETRIES + 1):
        attempt_count = attempt
        try:
            # using round number avoids event-name matching issues
            session = fastf1.get_session(season, round_number, session_code)
            session.load(laps=True, telemetry=EXTRACT_TELEMETRY, weather=True, messages=False)

            event_name_live = str(session.event["EventName"]) if "EventName" in session.event else event_name
            event_name_safe = safe_name(event_name_live)
            event_date_live = pd.to_datetime(session.event.get("EventDate", pd.NaT), errors="coerce")

            meta = {
                "Season": season,
                "RoundNumber": round_number,
                "EventName": event_name_live,
                "EventDate": event_date_live,
                "SessionCode": session_code,
                "SessionKey": session_key,
                "ExtractionTimestampUTC": RUN_TS,
            }

            results_df = build_results_table(session, meta)
            laps_df = build_laps_table(session, meta)
            telemetry_df = build_telemetry_features_table(session, meta)

            base_name = f"{season}_R{round_number:02d}_{event_name_safe}_{session_code}"

            saved_paths = []
            saved_paths += save_table_dual(results_df, results_dir / f"{base_name}_results")
            saved_paths += save_table_dual(laps_df, laps_dir / f"{base_name}_laps")
            saved_paths += save_table_dual(telemetry_df, telemetry_dir / f"{base_name}_telemetry_features")

            results_rows = len(results_df)
            laps_rows = len(laps_df)
            telemetry_rows = len(telemetry_df)
            status = "DONE"
            error_message = ""

            print(f"Saved files: {len(saved_paths)}")
            break

        except Exception as exc:
            error_message = str(exc)
            if attempt < MAX_RETRIES:
                wait_s = SLEEP_BETWEEN_RETRIES_SECONDS * attempt
                print(f"Attempt {attempt} failed. Retrying in {wait_s}s...")
                time.sleep(wait_s)
            else:
                print(f"Failed after {MAX_RETRIES} attempts.")

    elapsed_seconds = round(time.time() - session_start, 2)

    log_row = {
        "SessionKey": session_key,
        "Season": season,
        "RoundNumber": round_number,
        "EventName": event_name,
        "SessionCode": session_code,
        "Status": status,
        "AttemptCount": attempt_count,
        "ResultsRows": results_rows,
        "LapsRows": laps_rows,
        "TelemetryRows": telemetry_rows,
        "ElapsedSeconds": elapsed_seconds,
        "UpdatedAtUTC": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "ErrorMessage": error_message,
    }

    extraction_log_df = upsert_log_row(extraction_log_df, log_row)
    extraction_log_df = extraction_log_df[log_columns].sort_values(["Season", "RoundNumber", "SessionCode"])
    extraction_log_df.to_csv(EXTRACTION_LOG_PATH, index=False)

    print(
        f"Status={status} | time={elapsed_seconds}s | "
        f"rows: results={results_rows}, laps={laps_rows}, telemetry={telemetry_rows}"
    )

total_minutes = round((time.time() - overall_start) / 60, 2)
print(f"\nExtraction loop finished in {total_minutes} minutes.")
print("Log path:", EXTRACTION_LOG_PATH)

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...



[1/12] 2023_R01_Bahrain_Grand_Prix_Q


_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '11'

Saved files: 6
Status=DONE | time=6.8s | rows: results=20, laps=254, telemetry=254

[2/12] 2023_R01_Bahrain_Grand_Prix_R


_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for weather_data. Loading data...
_api           INFO 	Fetching weather data...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['1', '11'

...

Saved files: 6
Status=DONE | time=32.98s | rows: results=20, laps=927, telemetry=927

[11/12] 2025_R02_Chinese_Grand_Prix_Q


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

Saved files: 6
Status=DONE | time=15.07s | rows: results=20, laps=314, telemetry=314

[12/12] 2025_R02_Chinese_Grand_Prix_R


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

Saved files: 6
Status=DONE | time=29.66s | rows: results=20, laps=1065, telemetry=1065

Extraction loop finished in 4.3 minutes.
Log path: c:\Users\rdgonzaga\Documents\GitHub\Personal\f1-prediction\data\raw\logs\extraction_log.csv


## Step 5: Consolidate Session Files Into Master Tables

We combine all session-level files into three master datasets:

- results_combined
- laps_combined
- telemetry_features_combined

We save each combined table in both Parquet and CSV.


In [17]:
def load_many(paths, reader_fn):
    frames = []
    for p in paths:
        try:
            frames.append(reader_fn(p))
        except Exception as exc:
            print(f"Could not read {p.name}: {exc}")
    return frames


def consolidate_table(folder, file_suffix, combined_name):
    parquet_files = sorted(folder.glob(f"*_{file_suffix}.parquet"))
    csv_files = sorted(folder.glob(f"*_{file_suffix}.csv"))

    if parquet_files:
        frames = load_many(parquet_files, pd.read_parquet)
        source = "parquet"
    elif csv_files:
        frames = load_many(csv_files, pd.read_csv)
        source = "csv"
    else:
        print(f"No files found for {combined_name}.")
        return pd.DataFrame()

    if len(frames) == 0:
        print(f"Found files but none loaded for {combined_name}.")
        return pd.DataFrame()

    combined_df = pd.concat(frames, ignore_index=True)
    save_table_dual(combined_df, raw_dir / combined_name)
    print(f"{combined_name}: {len(combined_df):,} rows from {len(frames)} {source} files")
    return combined_df


results_combined = consolidate_table(results_dir, "results", "results_combined")
laps_combined = consolidate_table(laps_dir, "laps", "laps_combined")
telemetry_combined = consolidate_table(telemetry_dir, "telemetry_features", "telemetry_features_combined")

if not results_combined.empty:
    display(results_combined.head(5))
if not laps_combined.empty:
    display(laps_combined.head(5))
if not telemetry_combined.empty:
    display(telemetry_combined.head(5))

results_combined: 240 rows from 12 parquet files
laps_combined: 7,798 rows from 12 parquet files
telemetry_features_combined: 7,798 rows from 12 parquet files


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps,Q1Seconds,Q2Seconds,Q3Seconds,TimeSeconds,Season,RoundNumber,EventName,EventDate,SessionCode,SessionKey,ExtractionTimestampUTC
0,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,https://www.formula1.com/content/dam/fom-websi...,NED,1.0,,NaN,0 days 00:01:31.295000,0 days 00:01:30.503000,0 days 00:01:29.708000,NaT,,NaN,NaN,91.295,90.503,89.708,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
1,11,S PEREZ,PER,perez,Red Bull Racing,3671C6,red_bull,Sergio,Perez,Sergio Perez,https://www.formula1.com/content/dam/fom-websi...,MEX,2.0,,NaN,0 days 00:01:31.479000,0 days 00:01:30.746000,0 days 00:01:29.846000,NaT,,NaN,NaN,91.479,90.746,89.846,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
2,16,C LECLERC,LEC,leclerc,Ferrari,F91536,ferrari,Charles,Leclerc,Charles Leclerc,https://www.formula1.com/content/dam/fom-websi...,MON,3.0,,NaN,0 days 00:01:31.094000,0 days 00:01:30.282000,0 days 00:01:30,NaT,,NaN,NaN,91.094,90.282,90.000,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
3,55,C SAINZ,SAI,sainz,Ferrari,F91536,ferrari,Carlos,Sainz,Carlos Sainz,https://www.formula1.com/content/dam/fom-websi...,ESP,4.0,,NaN,0 days 00:01:30.993000,0 days 00:01:30.515000,0 days 00:01:30.154000,NaT,,NaN,NaN,90.993,90.515,90.154,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
4,14,F ALONSO,ALO,alonso,Aston Martin,358C75,aston_martin,Fernando,Alonso,Fernando Alonso,https://www.formula1.com/content/dam/fom-websi...,ESP,5.0,,NaN,0 days 00:01:31.158000,0 days 00:01:30.645000,0 days 00:01:30.336000,NaT,,NaN,NaN,91.158,90.645,90.336,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,LapTimeSeconds,Sector1TimeSeconds,Sector2TimeSeconds,Sector3TimeSeconds,DeltaToSessionBestLapSeconds,DeltaToDriverBestLapSeconds,Season,RoundNumber,EventName,EventDate,SessionCode,SessionKey,ExtractionTimestampUTC
0,0 days 00:27:20.459000,VER,1,NaT,1.0,1.0,0 days 00:18:59.843000,0 days 00:21:06.934000,NaT,NaT,NaT,NaT,NaT,NaT,204.0,NaN,NaN,178.0,False,SOFT,1.0,True,Red Bull Racing,0 days 00:18:59.843000,2023-03-04 15:04:00.840,15,NaN,None,,False,False,NaN,NaN,NaN,NaN,NaN,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
1,0 days 00:29:32.394000,VER,1,NaT,2.0,2.0,0 days 00:27:20.459000,NaT,NaT,0 days 00:00:53.666000,0 days 00:00:38.509000,NaT,0 days 00:28:53.926000,0 days 00:29:32.394000,219.0,217.0,288.0,141.0,False,SOFT,2.0,False,Red Bull Racing,0 days 00:27:20.459000,2023-03-04 15:12:21.456,1,NaN,None,,False,False,NaN,NaN,53.666,38.509,NaN,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
2,0 days 00:31:03.689000,VER,1,0 days 00:01:31.295000,3.0,2.0,NaT,NaT,0 days 00:00:29.152000,0 days 00:00:39.195000,0 days 00:00:22.948000,0 days 00:30:01.546000,0 days 00:30:40.741000,0 days 00:31:03.689000,240.0,270.0,288.0,322.0,True,SOFT,3.0,False,Red Bull Racing,0 days 00:29:32.394000,2023-03-04 15:14:33.391,1,NaN,None,,False,True,91.295,29.152,39.195,22.948,1.587,1.587,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
3,0 days 00:32:53.501000,VER,1,0 days 00:01:49.812000,4.0,2.0,NaT,0 days 00:32:51.749000,0 days 00:00:35.615000,0 days 00:00:44.953000,0 days 00:00:29.244000,0 days 00:31:39.304000,0 days 00:32:24.257000,0 days 00:32:53.501000,205.0,215.0,NaN,234.0,False,SOFT,4.0,False,Red Bull Racing,0 days 00:31:03.689000,2023-03-04 15:16:04.686,1,NaN,None,,False,False,109.812,35.615,44.953,29.244,20.104,20.104,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
4,0 days 00:40:05.688000,VER,1,NaT,5.0,3.0,0 days 00:37:48.525000,NaT,NaT,0 days 00:00:53.390000,0 days 00:00:45.107000,NaT,0 days 00:39:20.602000,0 days 00:40:05.820000,170.0,155.0,287.0,142.0,False,SOFT,1.0,True,Red Bull Racing,0 days 00:32:53.501000,2023-03-04 15:17:54.498,1,NaN,None,,False,False,NaN,NaN,53.390,45.107,NaN,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC


,Driver,DriverNumber,LapNumber,Stint,Compound,TyreLife,Samples,SpeedMean,SpeedStd,SpeedMin,SpeedMax,ThrottleMean,ThrottleStd,ThrottleMin,ThrottleMax,BrakeMean,BrakeStd,BrakeMin,BrakeMax,RPMMean,RPMStd,RPMMin,RPMMax,nGearMean,nGearStd,nGearMin,nGearMax,BoostActiveRatio,DistanceMax,LapTimeSeconds,Season,RoundNumber,EventName,EventDate,SessionCode,SessionKey,ExtractionTimestampUTC
0,VER,1,1.0,1.0,SOFT,1.0,1922,38.927680,65.837657,0.0,285.0,13.559313,29.215206,0.0,104.0,0.133195,0.339873,0.0,1.0,3090.417274,3976.651352,0.0,12059.0,0.937565,1.606291,0.0,8.0,1.0,5376.660278,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
1,VER,1,2.0,2.0,SOFT,2.0,497,137.881288,71.321116,31.0,291.0,41.456740,41.399200,0.0,100.0,0.205231,0.404278,0.0,1.0,9115.730382,2110.873580,3508.0,12260.0,3.070423,2.082248,1.0,8.0,1.0,5000.042778,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
2,VER,1,3.0,2.0,SOFT,3.0,345,212.997101,68.377394,68.0,323.0,69.646377,42.887456,0.0,100.0,0.200000,0.400581,0.0,1.0,10430.773913,1299.807788,5547.0,12069.0,5.243478,1.828034,2.0,8.0,1.0,5363.276111,91.295,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
3,VER,1,4.0,2.0,SOFT,4.0,421,175.396675,58.654451,60.0,299.0,47.672209,43.613857,0.0,100.0,0.206651,0.405385,0.0,1.0,9463.159145,1503.878491,5159.0,12078.0,4.551069,1.680527,2.0,8.0,1.0,5349.305278,109.812,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC
4,VER,1,5.0,3.0,SOFT,1.0,1627,45.052858,68.869127,0.0,289.0,26.515058,40.732593,0.0,104.0,0.232329,0.422448,0.0,1.0,3355.494161,4279.574278,0.0,11822.0,0.993239,1.672240,0.0,8.0,1.0,5395.390278,NaN,2023,1,Bahrain Grand Prix,2023-03-05,Q,2023_R01_Bahrain_Grand_Prix_Q,2026-04-12 19:21:40 UTC


## Step 6: Quality Checks and Coverage Report

We now validate:

- session completion status
- duplicate keys
- null rates on key columns
- telemetry join coverage against laps


In [18]:
def print_null_report(df, key_cols, name):
    if df is None or df.empty:
        print(f"{name}: empty table")
        return

    existing_cols = [c for c in key_cols if c in df.columns]
    if len(existing_cols) == 0:
        print(f"{name}: none of the requested key columns found")
        return

    null_pct = (df[existing_cols].isna().mean() * 100).round(2)
    report = null_pct.reset_index()
    report.columns = ["Column", "NullPct"]
    print(f"\n{name} null report")
    display(report)


def duplicate_count(df, subset_cols):
    if df is None or df.empty:
        return 0
    existing_cols = [c for c in subset_cols if c in df.columns]
    if len(existing_cols) != len(subset_cols):
        return np.nan
    return int(df.duplicated(subset=existing_cols).sum())


# Load latest log for final status summary
if EXTRACTION_LOG_PATH.exists():
    final_log_df = pd.read_csv(EXTRACTION_LOG_PATH)
else:
    final_log_df = extraction_log_df.copy()

print("Final status counts:")
display(final_log_df["Status"].value_counts(dropna=False))

coverage = (
    final_log_df.groupby(["Season", "SessionCode", "Status"])
    .size()
    .rename("Count")
    .reset_index()
    .sort_values(["Season", "SessionCode", "Status"])
)
display(coverage)

failed_df = final_log_df[final_log_df["Status"] == "ERROR"].copy()
print("Failed sessions:", len(failed_df))
if len(failed_df) > 0:
    display(failed_df[["SessionKey", "ErrorMessage"]].head(20))

# Duplicate checks
results_dup = duplicate_count(
    results_combined,
    ["Season", "RoundNumber", "SessionCode", "DriverNumber"]
)
laps_dup = duplicate_count(
    laps_combined,
    ["Season", "RoundNumber", "SessionCode", "Driver", "LapNumber"]
)
tele_dup = duplicate_count(
    telemetry_combined,
    ["Season", "RoundNumber", "SessionCode", "Driver", "LapNumber"]
)

print("\nDuplicate counts")
print("results_combined:", results_dup)
print("laps_combined:", laps_dup)
print("telemetry_combined:", tele_dup)

# Null reports on key columns
print_null_report(results_combined, ["Season", "RoundNumber", "SessionCode", "DriverNumber"], "results_combined")
print_null_report(laps_combined, ["Season", "RoundNumber", "SessionCode", "Driver", "LapNumber"], "laps_combined")
print_null_report(telemetry_combined, ["Season", "RoundNumber", "SessionCode", "Driver", "LapNumber"], "telemetry_combined")

# Join coverage: how many unique lap keys have telemetry
if not laps_combined.empty and not telemetry_combined.empty:
    join_cols = ["Season", "RoundNumber", "SessionCode", "Driver", "LapNumber"]
    if all(c in laps_combined.columns for c in join_cols) and all(c in telemetry_combined.columns for c in join_cols):
        lap_keys = laps_combined[join_cols].drop_duplicates()
        tele_keys = telemetry_combined[join_cols].drop_duplicates()

        merged = lap_keys.merge(tele_keys, on=join_cols, how="left", indicator=True)
        coverage_pct = (merged["_merge"].eq("both").mean() * 100).round(2)
        print(f"\nTelemetry key coverage vs laps: {coverage_pct}%")

Final status counts:


Status
DONE    12
Name: count, dtype: int64

,Season,SessionCode,Status,Count
0,2023,Q,DONE,2
1,2023,R,DONE,2
2,2024,Q,DONE,2
3,2024,R,DONE,2
4,2025,Q,DONE,2
5,2025,R,DONE,2


Failed sessions: 0

Duplicate counts
results_combined: 0
laps_combined: 0
telemetry_combined: 0

results_combined null report


,Column,NullPct
0,Season,0.0
1,RoundNumber,0.0
2,SessionCode,0.0
3,DriverNumber,0.0



laps_combined null report


,Column,NullPct
0,Season,0.0
1,RoundNumber,0.0
2,SessionCode,0.0
3,Driver,0.0
4,LapNumber,0.0



telemetry_combined null report


,Column,NullPct
0,Season,0.0
1,RoundNumber,0.0
2,SessionCode,0.0
3,Driver,0.0
4,LapNumber,0.0



Telemetry key coverage vs laps: 100.0%
